In [0]:
from pyspark.sql import functions as F

catalogo = "compras_rd"

# Crea esquemas si aún no existen (el Volume ya lo tienes)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.gold")

ruta = "/Volumes/compras_rd/bronze/raw_files/dominican_republic_api_2024.jsonl.gz"

df_bronze = (
    spark.read.json(ruta)
    .withColumn("_ingest_ts", F.current_timestamp())
    .withColumn("_source_file", F.lit(ruta))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.bronze.contratos_raw")
)

print(spark.table(f"{catalogo}.bronze.contratos_raw").count())

In [0]:
display(dbutils.fs.ls("/Volumes/compras_rd/bronze/raw_files/"))

Validación_de_la_ingesta_en_Bronze

In [0]:

# Definir catálogo, esquema y tabla
catalogo = "compras_rd"
esquema = "bronze"
tabla = "contratos_raw"

nombre_tabla = f"{catalogo}.{esquema}.{tabla}"

# 1. Verificar cantidad de registros
total_registros = spark.table(nombre_tabla).count()

print(f"Tabla: {nombre_tabla}")
print(f"Total de registros ingeridos: {total_registros:,}")

# 2. Consultar estructura de la tabla
print("\nEstructura de la tabla:")
display(spark.sql(f"DESCRIBE TABLE {nombre_tabla}"))

# 3. Visualizar los primeros registros
print("\nPrimeros registros:")
display(spark.table(nombre_tabla).limit(10))

# 4. Verificar metadatos de ingesta
print("\nInformación de ingesta:")
display(
    spark.table(nombre_tabla)
    .select("_ingest_ts", "_source_file")
    .distinct()
)
